# Module 11: MLflow Tracking for Feature Selection

## Learning Objectives

By completing this notebook, you will:
1. Set up an MLflow experiment dedicated to feature selection runs
2. Log selection results: selected features, importance scores, method parameters, and validation metrics
3. Compare multiple selection runs in the MLflow UI by querying the tracking server
4. Track feature set versions over time using a content-hash version registry
5. Build a complete selection audit trail that satisfies production and regulatory requirements

## Prerequisites
- Guide 03 (MLflow concepts, versioning, audit trails)
- Notebook 01 (pipeline building)
- Notebook 02 (drift monitoring — triggers that initiate selection runs)

## Estimated Time: 15 minutes

---

## 1. Setup: MLflow Experiment

We use a local SQLite backend for the tracking server. In production, replace `sqlite:///mlflow.db` with the URI of your shared MLflow server (e.g., `http://mlflow.company.internal:5000`).

The experiment name scopes all runs to a single project. Every selection method, every dataset version, and every parameter combination gets its own `run` within this experiment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import json
import hashlib
import datetime
import pathlib
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, SelectKBest, f_classif
from sklearn.linear_model import LassoCV
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils import resample
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import roc_auc_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

# --- MLflow setup ---
EXPERIMENT_NAME = "feature-selection-production"
mlflow.set_tracking_uri("sqlite:///mlflow_tracking.db")
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

# --- Load data ---
housing = fetch_california_housing(as_frame=True)
X = housing.frame.drop(columns=['MedHouseVal'])
y = (housing.frame['MedHouseVal'] > housing.frame['MedHouseVal'].median()).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, stratify=y_test
)

print(f"\nDataset: {X.shape[0]:,} total | Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

## 2. Logging a Selection Run

The `run_and_log_selection` function is the core integration point. It:
1. Runs feature selection
2. Trains a model on selected features
3. Logs everything to MLflow: parameters, metrics, artefacts, and the pipeline

The function returns the MLflow `run_id` so the run can be retrieved later for comparison.

In [ ]:
def hash_dataframe(df: pd.DataFrame) -> str:
    """Deterministic hash of a DataFrame (for dataset version tracking)."""
    return hashlib.sha256(
        pd.util.hash_pandas_object(df, index=True).values.tobytes()
    ).hexdigest()[:16]


def run_and_log_selection(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_val: pd.DataFrame,
    y_val: np.ndarray,
    selector,
    selector_name: str,
    selector_params: dict,
    model_class=GradientBoostingClassifier,
    model_params: dict | None = None,
    tags: dict | None = None,
) -> str:
    """
    Run feature selection, train a model, and log everything to MLflow.

    Parameters
    ----------
    selector : sklearn selector
        Must implement fit(X, y) and get_support().
    selector_name : str
        Human-readable name for the selection method.
    selector_params : dict
        Hyperparameters of the selector (logged as MLflow params).
    tags : dict | None
        Additional MLflow tags (e.g. triggered_by, dataset_version).

    Returns
    -------
    str
        MLflow run_id (first 8 chars for readability).
    """
    model_params = model_params or {'n_estimators': 100, 'max_depth': 3, 'random_state': 42}
    dataset_hash = hash_dataframe(X_train)

    tmp_dir = pathlib.Path('mlflow_tmp')
    tmp_dir.mkdir(exist_ok=True)

    with mlflow.start_run(tags=tags or {}) as run:
        run_id = run.info.run_id

        # --- 1. Log provenance parameters ---
        mlflow.log_param('selector_name',    selector_name)
        mlflow.log_param('dataset_hash',     dataset_hash)
        mlflow.log_param('n_features_total', X_train.shape[1])
        mlflow.log_param('n_train',          len(X_train))
        mlflow.log_param('selection_date',   datetime.date.today().isoformat())
        for k, v in selector_params.items():
            mlflow.log_param(f'selector.{k}', v)
        for k, v in model_params.items():
            mlflow.log_param(f'model.{k}', v)

        # --- 2. Run selector ---
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_train)
        X_va_scaled = scaler.transform(X_val)

        # Wrap in DataFrame to preserve column names
        X_tr_df = pd.DataFrame(X_tr_scaled, columns=X_train.columns)
        X_va_df = pd.DataFrame(X_va_scaled, columns=X_val.columns)

        selector.fit(X_tr_df, y_train)
        support      = selector.get_support()
        selected_names = X_train.columns[support].tolist()
        n_selected   = len(selected_names)

        # --- 3. Log selection metrics ---
        mlflow.log_metric('n_features_selected', n_selected)
        mlflow.log_metric('selection_ratio', n_selected / X_train.shape[1])

        # --- 4. Train model on selected features ---
        X_tr_sel = X_tr_df.iloc[:, support]
        X_va_sel = X_va_df.iloc[:, support]

        model = model_class(**model_params)
        model.fit(X_tr_sel, y_train)

        y_prob_val = model.predict_proba(X_va_sel)[:, 1]
        val_auc = roc_auc_score(y_val, y_prob_val)

        mlflow.log_metric('val_roc_auc', val_auc)

        # --- 5. Log artefacts ---
        # Feature list as JSON (avoids MLflow 500-param limit)
        features_file = tmp_dir / 'selected_features.json'
        features_file.write_text(json.dumps({
            'selected_features': selected_names,
            'n_selected': n_selected,
            'selector': selector_name,
            'dataset_hash': dataset_hash,
            'val_roc_auc': round(val_auc, 6),
        }, indent=2))
        mlflow.log_artifact(str(features_file))

        # Feature scores (if available)
        if hasattr(selector, 'scores_'):
            scores_df = pd.DataFrame({
                'feature':  X_train.columns,
                'score':    selector.scores_,
                'selected': support,
            }).sort_values('score', ascending=False)
            scores_file = tmp_dir / 'feature_scores.csv'
            scores_df.to_csv(scores_file, index=False)
            mlflow.log_artifact(str(scores_file))
        elif hasattr(selector, 'estimator_') and hasattr(selector.estimator_, 'feature_importances_'):
            importances = selector.estimator_.feature_importances_
            imp_df = pd.DataFrame({
                'feature':    X_train.columns,
                'importance': importances,
                'selected':   support,
            }).sort_values('importance', ascending=False)
            imp_file = tmp_dir / 'feature_importances.csv'
            imp_df.to_csv(imp_file, index=False)
            mlflow.log_artifact(str(imp_file))

        # Log full pipeline as MLflow model
        full_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model',  model),
        ])
        # Refit scaler+model so the logged pipeline is self-contained
        X_tr_sel_raw = X_train.iloc[:, support]
        full_pipeline.fit(X_tr_sel_raw, y_train)
        mlflow.sklearn.log_model(full_pipeline, 'pipeline')

        print(f"  Run {run_id[:8]}: {selector_name:20s} → "
              f"{n_selected}/{X_train.shape[1]} features, Val AUC={val_auc:.4f}")

    return run_id[:8]


print("run_and_log_selection defined.")

## 3. Run Multiple Selection Methods

We run three different selection methods and log each as a separate MLflow run. This builds the comparison database that we query in the next section.

In [ ]:
print("Logging selection runs to MLflow...")
print()

run_ids = {}

# --- Run 1: SelectKBest (f_classif) ---
run_ids['selectkbest_k5'] = run_and_log_selection(
    X_train, y_train, X_val, y_val,
    selector=SelectKBest(score_func=f_classif, k=5),
    selector_name='selectkbest',
    selector_params={'score_func': 'f_classif', 'k': 5},
    tags={'method_family': 'filter', 'triggered_by': 'manual'},
)

# --- Run 2: SelectKBest with k=3 ---
run_ids['selectkbest_k3'] = run_and_log_selection(
    X_train, y_train, X_val, y_val,
    selector=SelectKBest(score_func=f_classif, k=3),
    selector_name='selectkbest',
    selector_params={'score_func': 'f_classif', 'k': 3},
    tags={'method_family': 'filter', 'triggered_by': 'manual'},
)

# --- Run 3: SelectFromModel (RandomForest, threshold=mean) ---
run_ids['rf_mean'] = run_and_log_selection(
    X_train, y_train, X_val, y_val,
    selector=SelectFromModel(
        estimator=RandomForestClassifier(n_estimators=100, random_state=42),
        threshold='mean',
    ),
    selector_name='selectfrommodel_rf',
    selector_params={'base_estimator': 'RandomForest', 'n_estimators': 100, 'threshold': 'mean'},
    tags={'method_family': 'embedded', 'triggered_by': 'psi_drift'},
)

# --- Run 4: SelectFromModel (RandomForest, threshold=median) ---
run_ids['rf_median'] = run_and_log_selection(
    X_train, y_train, X_val, y_val,
    selector=SelectFromModel(
        estimator=RandomForestClassifier(n_estimators=100, random_state=42),
        threshold='median',
    ),
    selector_name='selectfrommodel_rf',
    selector_params={'base_estimator': 'RandomForest', 'n_estimators': 100, 'threshold': 'median'},
    tags={'method_family': 'embedded', 'triggered_by': 'psi_drift'},
)

print()
print(f"Logged {len(run_ids)} runs to MLflow experiment '{EXPERIMENT_NAME}'")

## 4. Compare Runs in MLflow

Query the MLflow tracking server to retrieve all runs from the experiment and compare them. This is the programmatic equivalent of the MLflow UI comparison view.

In [ ]:
def compare_selection_runs(experiment_name: str) -> pd.DataFrame:
    """
    Retrieve all runs from an experiment and return a comparison DataFrame.

    Sorted by val_roc_auc descending.
    """
    client = mlflow.tracking.MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found")

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['metrics.val_roc_auc DESC'],
    )

    records = []
    for run in runs:
        records.append({
            'run_id':        run.info.run_id[:8],
            'selector':      run.data.params.get('selector_name', '—'),
            'n_selected':    int(run.data.metrics.get('n_features_selected', 0)),
            'sel_ratio':     round(run.data.metrics.get('selection_ratio', 0), 3),
            'val_roc_auc':   round(run.data.metrics.get('val_roc_auc', float('nan')), 4),
            'dataset_hash':  run.data.params.get('dataset_hash', '—'),
            'date':          pd.Timestamp(run.info.start_time, unit='ms').strftime('%Y-%m-%d'),
            'tags':          str(run.data.tags),
        })

    return pd.DataFrame(records)


comparison = compare_selection_runs(EXPERIMENT_NAME)
print(f"All runs in experiment '{EXPERIMENT_NAME}':")
print()
print(comparison[['run_id', 'selector', 'n_selected', 'sel_ratio', 'val_roc_auc', 'date']]
      .to_string(index=False))
print()
print(f"Best run: {comparison.iloc[0]['run_id']} "
      f"({comparison.iloc[0]['selector']}, {comparison.iloc[0]['n_selected']} features, "
      f"AUC={comparison.iloc[0]['val_roc_auc']})")

In [ ]:
# Visualise the comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1: AUC by run
run_labels = [f"{r['run_id']}\n{r['selector']} k={r['n_selected']}" for _, r in comparison.iterrows()]
bar_colors = ['#2ca02c' if i == 0 else '#1f77b4' for i in range(len(comparison))]
ax1.barh(run_labels, comparison['val_roc_auc'], color=bar_colors, edgecolor='white', alpha=0.85)
ax1.set_xlabel('Validation ROC AUC', fontsize=12)
ax1.set_title('Selection Runs by AUC', fontsize=12)
ax1.axvline(x=comparison['val_roc_auc'].max(), color='green', linestyle='--', alpha=0.5)
ax1.set_xlim(comparison['val_roc_auc'].min() - 0.02, comparison['val_roc_auc'].max() + 0.02)
for i, (_, row) in enumerate(comparison.iterrows()):
    ax1.text(row['val_roc_auc'] + 0.001, i, f"{row['val_roc_auc']:.4f}",
              va='center', fontsize=9)

# Chart 2: AUC vs number of features
ax2.scatter(comparison['n_selected'], comparison['val_roc_auc'],
             c=bar_colors, s=120, edgecolors='black', linewidth=0.5, zorder=3)
for _, row in comparison.iterrows():
    ax2.annotate(f"{row['run_id']}",
                  (row['n_selected'], row['val_roc_auc']),
                  textcoords='offset points', xytext=(5, 3), fontsize=8)
ax2.set_xlabel('Number of Selected Features', fontsize=12)
ax2.set_ylabel('Validation ROC AUC', fontsize=12)
ax2.set_title('AUC vs. Feature Count (Pareto trade-off)', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.suptitle('MLflow Run Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Feature Set Versioning

Each selection run produces a specific feature set. We version it using a deterministic content hash: the same inputs always produce the same version ID. This enables rollback, comparison, and audit.

In [ ]:
import dataclasses
from dataclasses import dataclass, field, asdict


@dataclass
class FeatureSelectionVersion:
    """
    Immutable record of a feature selection event.
    Version ID is a SHA-256 hash of the selection fingerprint.
    """
    version_id:        str
    selected_features: list
    n_features_total:  int
    method:            str
    params:            dict
    dataset_hash:      str
    random_seed:       int
    selection_date:    str
    val_roc_auc:       float
    mlflow_run_id:     str | None = None

    @classmethod
    def create(
        cls,
        selected_features: list,
        n_features_total: int,
        method: str,
        params: dict,
        dataset_hash: str,
        random_seed: int,
        val_roc_auc: float,
        mlflow_run_id: str | None = None,
    ) -> 'FeatureSelectionVersion':
        # Deterministic fingerprint — same inputs → same hash
        fingerprint = json.dumps({
            'features':     sorted(selected_features),
            'dataset_hash': dataset_hash,
            'method':       method,
            'params':       params,
            'seed':         random_seed,
        }, sort_keys=True)
        version_id = hashlib.sha256(fingerprint.encode()).hexdigest()[:12]

        return cls(
            version_id=version_id,
            selected_features=sorted(selected_features),
            n_features_total=n_features_total,
            method=method,
            params=params,
            dataset_hash=dataset_hash,
            random_seed=random_seed,
            selection_date=datetime.date.today().isoformat(),
            val_roc_auc=round(val_roc_auc, 6),
            mlflow_run_id=mlflow_run_id,
        )

    def to_json(self, path: str) -> None:
        pathlib.Path(path).write_text(json.dumps(asdict(self), indent=2))

    @classmethod
    def from_json(cls, path: str) -> 'FeatureSelectionVersion':
        return cls(**json.loads(pathlib.Path(path).read_text()))


class SelectionVersionRegistry:
    """File-based registry of all feature selection versions."""

    def __init__(self, registry_dir: str = 'selection_versions/'):
        self.registry_dir = pathlib.Path(registry_dir)
        self.registry_dir.mkdir(parents=True, exist_ok=True)

    def save(self, version: FeatureSelectionVersion) -> None:
        path = self.registry_dir / f"{version.selection_date}_{version.version_id}.json"
        version.to_json(str(path))
        print(f"Saved version {version.version_id} ({version.method}, "
              f"{len(version.selected_features)} features, AUC={version.val_roc_auc:.4f})")

    def load(self, version_id: str) -> FeatureSelectionVersion:
        matches = list(self.registry_dir.glob(f'*{version_id}*.json'))
        if not matches:
            raise KeyError(f"Version {version_id} not found in registry")
        return FeatureSelectionVersion.from_json(str(matches[0]))

    def list_versions(self) -> pd.DataFrame:
        versions = [
            FeatureSelectionVersion.from_json(str(f))
            for f in sorted(self.registry_dir.glob('*.json'))
        ]
        if not versions:
            return pd.DataFrame()
        return pd.DataFrame([{
            'version_id': v.version_id,
            'date':       v.selection_date,
            'method':     v.method,
            'n_selected': len(v.selected_features),
            'val_auc':    v.val_roc_auc,
            'mlflow_run': v.mlflow_run_id or '—',
        } for v in versions]).sort_values('date', ascending=False)

    def get_current(self) -> FeatureSelectionVersion:
        df = self.list_versions()
        if df.empty:
            raise RuntimeError("No versions in registry")
        return self.load(df.iloc[0]['version_id'])


print("Version classes defined.")

In [ ]:
# Create versions for the runs we logged above
registry = SelectionVersionRegistry('selection_versions/')
dataset_hash = hash_dataframe(X_train)

print("Creating and saving selection versions:")
print()

# Version 1: SelectKBest k=5
sel_kbest5 = SelectKBest(score_func=f_classif, k=5)
sel_kbest5.fit(StandardScaler().fit_transform(X_train), y_train)
kbest5_features = X_train.columns[sel_kbest5.get_support()].tolist()
v1 = FeatureSelectionVersion.create(
    selected_features=kbest5_features,
    n_features_total=X_train.shape[1],
    method='selectkbest',
    params={'score_func': 'f_classif', 'k': 5},
    dataset_hash=dataset_hash,
    random_seed=42,
    val_roc_auc=comparison[comparison['selector'] == 'selectkbest']['val_roc_auc'].max(),
    mlflow_run_id=run_ids.get('selectkbest_k5'),
)
registry.save(v1)

# Version 2: RF median
sel_rf = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=100, random_state=42),
    threshold='median',
)
sel_rf.fit(StandardScaler().fit_transform(X_train), y_train)
rf_features = X_train.columns[sel_rf.get_support()].tolist()
v2 = FeatureSelectionVersion.create(
    selected_features=rf_features,
    n_features_total=X_train.shape[1],
    method='selectfrommodel_rf',
    params={'base_estimator': 'RandomForest', 'n_estimators': 100, 'threshold': 'median'},
    dataset_hash=dataset_hash,
    random_seed=42,
    val_roc_auc=comparison[comparison['selector'] == 'selectfrommodel_rf']['val_roc_auc'].max(),
    mlflow_run_id=run_ids.get('rf_median'),
)
registry.save(v2)

print()
print("Registry contents:")
print(registry.list_versions().to_string(index=False))

## 6. Feature Set Version Comparison Over Time

As feature sets evolve across re-selection events, tracking which features are stable (selected every time) vs. volatile (selected sometimes) helps identify the core signal vs. noise-sensitive features.

In [ ]:
def compare_version_feature_sets(registry: SelectionVersionRegistry) -> pd.DataFrame:
    """
    Create a feature × version presence matrix.

    Returns a DataFrame where rows are features, columns are version IDs,
    and values are 1 (selected) or 0 (not selected).
    """
    version_df = registry.list_versions()
    versions = [registry.load(vid) for vid in version_df['version_id']]

    # Get union of all features across all versions
    all_features = sorted(set(
        feat for v in versions for feat in v.selected_features
    ))

    # Build presence matrix
    matrix = {}
    for v in versions:
        selected_set = set(v.selected_features)
        matrix[f"{v.version_id}\n({v.method})"] = [
            1 if feat in selected_set else 0
            for feat in all_features
        ]

    df = pd.DataFrame(matrix, index=all_features)
    df['n_versions_selected'] = df.sum(axis=1)
    return df.sort_values('n_versions_selected', ascending=False)


presence_matrix = compare_version_feature_sets(registry)
print("Feature presence across versions:")
print(presence_matrix.to_string())
print()

# Visualise
fig, ax = plt.subplots(figsize=(10, max(4, len(presence_matrix) * 0.6)))
version_cols = [c for c in presence_matrix.columns if c != 'n_versions_selected']
im = ax.imshow(presence_matrix[version_cols].values, cmap='RdYlGn', aspect='auto',
                vmin=0, vmax=1)
ax.set_xticks(range(len(version_cols)))
ax.set_xticklabels(version_cols, fontsize=9)
ax.set_yticks(range(len(presence_matrix.index)))
ax.set_yticklabels(presence_matrix.index, fontsize=10)
ax.set_title('Feature Selection Stability Across Versions\n(green = selected, red = not selected)',
              fontsize=11)
plt.colorbar(im, ax=ax, label='Selected (1) / Not selected (0)')
plt.tight_layout()
plt.show()

## 7. Building the Audit Trail

An audit trail records every selection event with full provenance: what was selected, when, why it was triggered, who approved it, and which MLflow run produced it.

In [ ]:
def create_audit_record(
    version: FeatureSelectionVersion,
    triggered_by: str,
    trigger_reason: str,
    approved_by: str | None = None,
) -> dict:
    """
    Create a structured audit record.

    Parameters
    ----------
    triggered_by : str
        'scheduled' | 'psi_trigger' | 'ks_trigger' | 'performance' | 'manual'
    trigger_reason : str
        Human-readable description of why re-selection was triggered.
    approved_by : str | None
        User ID of approver (required in regulated environments).
    """
    return {
        'audit_timestamp':     datetime.datetime.utcnow().isoformat() + 'Z',  # UTC
        'version_id':          version.version_id,
        'selection_date':      version.selection_date,
        'method':              version.method,
        'params':              version.params,
        'n_features_total':    version.n_features_total,
        'n_features_selected': len(version.selected_features),
        'selected_features':   version.selected_features,
        'val_roc_auc':         version.val_roc_auc,
        'dataset_hash':        version.dataset_hash,
        'random_seed':         version.random_seed,
        'mlflow_run_id':       version.mlflow_run_id,
        'triggered_by':        triggered_by,
        'trigger_reason':      trigger_reason,
        'approved_by':         approved_by or 'PENDING',
    }


class AuditTrail:
    """Append-only audit log of all selection events."""

    def __init__(self, audit_path: str = 'selection_audit.jsonl'):
        self.audit_path = pathlib.Path(audit_path)

    def append(self, record: dict) -> None:
        """Append a single audit record (one JSON object per line)."""
        with open(self.audit_path, 'a') as f:
            f.write(json.dumps(record) + '\n')

    def read_all(self) -> pd.DataFrame:
        """Read the full audit trail as a DataFrame."""
        if not self.audit_path.exists():
            return pd.DataFrame()
        records = [json.loads(line) for line in self.audit_path.read_text().splitlines()]
        return pd.DataFrame(records)


# Build audit records for the two versions we created
audit = AuditTrail('selection_audit.jsonl')

audit.append(create_audit_record(
    version=v1,
    triggered_by='manual',
    trigger_reason='Initial feature selection for baseline model deployment',
    approved_by='mlops-team',
))

audit.append(create_audit_record(
    version=v2,
    triggered_by='psi_trigger',
    trigger_reason=(
        'PSI trigger: 3/8 features critical (drift_mean_shift PSI=0.34, '
        'drift_variance PSI=0.28, drift_combined PSI=0.22)'
    ),
    approved_by='model-validation',
))

audit_df = audit.read_all()
print("Audit trail (production format):")
print()
for _, row in audit_df.iterrows():
    print(f"{'='*60}")
    print(f"Timestamp:       {row['audit_timestamp']}")
    print(f"Version ID:      {row['version_id']}")
    print(f"Method:          {row['method']}")
    print(f"Features:        {row['selected_features']}")
    print(f"Val AUC:         {row['val_roc_auc']}")
    print(f"Dataset hash:    {row['dataset_hash']}")
    print(f"MLflow run:      {row['mlflow_run_id']}")
    print(f"Triggered by:    {row['triggered_by']}")
    print(f"Trigger reason:  {row['trigger_reason']}")
    print(f"Approved by:     {row['approved_by']}")

## 8. Exercise: Log Wasserstein Drift Metrics as MLflow Metrics

**Task:** Write a function `log_drift_metrics_to_mlflow` that:
1. Takes reference data, current production data, and a run name as inputs
2. Computes PSI for each feature
3. Computes the KS statistic for each feature
4. Logs each metric to MLflow as `psi_{feature_name}` and `ks_{feature_name}`
5. Also logs summary metrics: `n_features_psi_critical`, `n_features_ks_drifted`, `mean_psi`
6. Returns the MLflow run_id

This would be called automatically by the drift monitoring pipeline every time it checks for drift — building a complete time series of drift metrics in MLflow.

**Hint:** Use `mlflow.log_metric(key, value)` inside a `with mlflow.start_run()` block.

In [ ]:
def compute_psi(reference, current, n_bins=10, eps=1e-6):
    """PSI from drift monitoring notebook (reproduced here for completeness)."""
    bin_edges = np.percentile(reference, np.linspace(0, 100, n_bins + 1))
    bin_edges[0] -= 1e-10
    bin_edges[-1] += 1e-10
    ref_counts = np.histogram(reference, bins=bin_edges)[0]
    cur_counts = np.histogram(current,   bins=bin_edges)[0]
    ref_props = (ref_counts / ref_counts.sum()) + eps
    cur_props = (cur_counts / cur_counts.sum()) + eps
    ref_props /= ref_props.sum()
    cur_props /= cur_props.sum()
    return float(np.sum((cur_props - ref_props) * np.log(cur_props / ref_props)))


def log_drift_metrics_to_mlflow(
    reference: pd.DataFrame,
    current: pd.DataFrame,
    run_name: str,
    month: int,
) -> str:
    """
    Compute PSI and KS drift metrics and log them to MLflow.

    Parameters
    ----------
    reference : pd.DataFrame
        Training distribution reference data.
    current : pd.DataFrame
        Current production data.
    run_name : str
        MLflow run name (e.g. 'drift_check_month_06').
    month : int
        Production month index (logged as a metric for time series tracking).

    Returns
    -------
    str
        MLflow run_id (first 8 chars).
    """
    # TODO: Implement
    # 1. Open a new MLflow run with run_name and tags={'type': 'drift_monitoring'}
    # 2. Log month as a param: mlflow.log_param('month', month)
    # 3. For each feature:
    #    a. Compute PSI: psi_val = compute_psi(reference[col].values, current[col].values)
    #    b. Compute KS: stat, _ = ks_2samp(reference[col].values, current[col].values)
    #    c. Log: mlflow.log_metric(f'psi_{col}', psi_val)
    #       and: mlflow.log_metric(f'ks_{col}', stat)
    # 4. Log summary metrics:
    #    - n_features_psi_critical: count of features with PSI > 0.20
    #    - n_features_ks_drifted:   count with KS stat > 0.10 (simple threshold)
    #    - mean_psi
    # 5. Return the run_id[:8]
    raise NotImplementedError("Implement log_drift_metrics_to_mlflow")


# Self-check tests
def test_drift_logging():
    from scipy.stats import ks_2samp
    import numpy as np
    import pandas as pd

    # Create simple test data
    rng = np.random.default_rng(0)
    ref = pd.DataFrame({'a': rng.normal(0, 1, 200), 'b': rng.normal(0, 1, 200)})
    cur = pd.DataFrame({'a': rng.normal(1, 1, 200), 'b': rng.normal(0, 1, 200)})  # 'a' drifted

    run_id = log_drift_metrics_to_mlflow(ref, cur, run_name='test_drift', month=0)

    assert isinstance(run_id, str) and len(run_id) >= 8, "run_id must be at least 8 chars"

    # Verify metrics were logged
    client = mlflow.tracking.MlflowClient()
    full_run_id = next(
        r.info.run_id for r in client.search_runs(
            experiment_ids=[client.get_experiment_by_name(EXPERIMENT_NAME).experiment_id],
            order_by=['attribute.start_time DESC'],
        )
        if r.info.run_id.startswith(run_id)
    )
    metrics = client.get_run(full_run_id).data.metrics
    assert 'psi_a' in metrics, "Expected metric 'psi_a' to be logged"
    assert 'psi_b' in metrics, "Expected metric 'psi_b' to be logged"
    assert 'ks_a'  in metrics, "Expected metric 'ks_a' to be logged"
    assert 'mean_psi' in metrics, "Expected 'mean_psi' summary metric"
    assert metrics['psi_a'] > metrics['psi_b'], (
        "'a' drifted more than 'b' — PSI for 'a' should be higher"
    )

    print(f"Test passed! run_id={run_id}, psi_a={metrics['psi_a']:.4f}, psi_b={metrics['psi_b']:.4f}")

# Uncomment to run after implementing:
# test_drift_logging()

## 9. Summary

### Key Takeaways

1. **Log everything**: Parameters (inputs), metrics (outputs), artefacts (feature lists), and the pipeline model. Any of these omitted makes debugging or auditing impossible later.

2. **Content-hash version IDs**: The same inputs always produce the same version ID. Different inputs always produce different IDs. This gives you deterministic identity without relying on labels.

3. **Audit trails are append-only**: Never delete or modify audit records. Append a correction record if something went wrong, but preserve the original. JSONL format (one JSON object per line) is ideal for this.

4. **Connect MLflow runs to version records**: Store the `mlflow_run_id` in the version JSON. This creates a two-way link: from the version registry you can find the MLflow run; from the MLflow run you can find the version.

5. **Log drift metrics alongside selection runs**: Using the same MLflow experiment for both selection runs and drift monitoring creates a single dashboard that shows the complete lifecycle: drift detected → selection triggered → new features deployed → drift monitored again.

### Module 11 Complete

You have now built the complete production feature selection infrastructure:
- **Notebook 01**: Pipeline with stability selection — the deployment artefact
- **Notebook 02**: Drift monitoring with PSI/KS — the early warning system
- **Notebook 03**: MLflow tracking and versioning — the audit and governance layer

### Additional Resources
- MLflow documentation: https://mlflow.org/docs/latest/index.html
- Zaharia et al. (2018): Accelerating the Machine Learning Lifecycle with MLflow
- Sculley et al. (2015): Hidden Technical Debt in Machine Learning Systems